## Goal: Prepare a clean, model-ready dataset by applying transformations identified during EDA in a reproducible and defensible way.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('/Users/eugene/Desktop/Emory/Projects/Travelers/data/raw/Training_TriGuard.csv')
df.head()

,subrogation,claim_number,year_of_born,gender,email_or_tel_available,safety_rating,annual_income,high_education_ind,address_change_ind,living_status,...,claim_est_payout,vehicle_made_year,vehicle_category,vehicle_price,vehicle_color,vehicle_weight,age_of_DL,accident_type,in_network_bodyshop,vehicle_mileage
0,1.0,6090851,1990.0,F,0.0,75.0,70966.0,1.0,1.0,Rent,...,3218.84,2021.0,Large,16272.12725,red,21620.79697,25.0,multi_vehicle_clear,no,75421.0
1,0.0,4653734,1972.0,F,1.0,94.0,79723.0,1.0,1.0,Rent,...,1338.52,2025.0,Medium,34102.78197,silver,10840.58520,23.0,multi_vehicle_clear,yes,31988.0
2,0.0,1014777,2003.0,F,1.0,76.0,41527.0,1.0,1.0,Own,...,3540.05,2022.0,Compact,15000.00000,silver,24318.12282,23.0,multi_vehicle_unclear,yes,60876.0
3,1.0,8101873,1983.0,F,1.0,54.0,42099.0,1.0,1.0,Rent,...,1507.94,2025.0,Medium,16984.45295,white,36958.26656,23.0,multi_vehicle_unclear,yes,152772.0
4,0.0,5081870,1985.0,F,1.0,54.0,47206.0,1.0,1.0,Own,...,5080.63,2021.0,Compact,46545.72863,blue,11779.17422,17.0,multi_vehicle_clear,yes,41151.0


In [3]:
# Driver age
df["claim_date"] = pd.to_datetime(df["claim_date"], errors="coerce")
df["driver_age"] = df["claim_date"].dt.year - df["year_of_born"]

In [4]:
df.head()

,subrogation,claim_number,year_of_born,gender,email_or_tel_available,safety_rating,annual_income,high_education_ind,address_change_ind,living_status,...,vehicle_made_year,vehicle_category,vehicle_price,vehicle_color,vehicle_weight,age_of_DL,accident_type,in_network_bodyshop,vehicle_mileage,driver_age
0,1.0,6090851,1990.0,F,0.0,75.0,70966.0,1.0,1.0,Rent,...,2021.0,Large,16272.12725,red,21620.79697,25.0,multi_vehicle_clear,no,75421.0,26.0
1,0.0,4653734,1972.0,F,1.0,94.0,79723.0,1.0,1.0,Rent,...,2025.0,Medium,34102.78197,silver,10840.58520,23.0,multi_vehicle_clear,yes,31988.0,43.0
2,0.0,1014777,2003.0,F,1.0,76.0,41527.0,1.0,1.0,Own,...,2022.0,Compact,15000.00000,silver,24318.12282,23.0,multi_vehicle_unclear,yes,60876.0,12.0
3,1.0,8101873,1983.0,F,1.0,54.0,42099.0,1.0,1.0,Rent,...,2025.0,Medium,16984.45295,white,36958.26656,23.0,multi_vehicle_unclear,yes,152772.0,32.0
4,0.0,5081870,1985.0,F,1.0,54.0,47206.0,1.0,1.0,Own,...,2021.0,Compact,46545.72863,blue,11779.17422,17.0,multi_vehicle_clear,yes,41151.0,31.0


In [5]:
print(min(df.driver_age))
print(max(df.driver_age))

8.0
241.0


In [6]:
# Applying bounds to the driver age
df.loc[(df["driver_age"] < 16) | (df["driver_age"] > 100), "driver_age"] = np.nan

In [7]:
df[df["driver_age"].isna()].shape

(651, 30)

In [8]:
# Extracting year and month from claim date
df["claim_year"] = df["claim_date"].dt.year
df["claim_month"] = df["claim_date"].dt.month
df[["claim_date", "claim_year", "claim_month"]].head()

,claim_date,claim_year,claim_month
0,2016-12-04,2016.0,12.0
1,2015-04-25,2015.0,4.0
2,2015-06-22,2015.0,6.0
3,2015-03-02,2015.0,3.0
4,2016-01-12,2016.0,1.0


In [9]:
# Dropping columns that will not be used in modelling
DROP_COLS = [
    "claim_number",
    "year_of_born",
    "claim_date",
]

df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

In [10]:
# Log transform
df["log_claim_est_payout"] = np.log1p(df["claim_est_payout"])
df["log_vehicle_price"] = np.log1p(df["vehicle_price"])
df.head()

,subrogation,gender,email_or_tel_available,safety_rating,annual_income,high_education_ind,address_change_ind,living_status,zip_code,claim_day_of_week,...,vehicle_weight,age_of_DL,accident_type,in_network_bodyshop,vehicle_mileage,driver_age,claim_year,claim_month,log_claim_est_payout,log_vehicle_price
0,1.0,F,0.0,75.0,70966.0,1.0,1.0,Rent,80040.0,Saturday,...,21620.79697,25.0,multi_vehicle_clear,no,75421.0,26.0,2016.0,12.0,8.077087,9.697270
1,0.0,F,1.0,94.0,79723.0,1.0,1.0,Rent,80030.0,Wednesday,...,10840.58520,23.0,multi_vehicle_clear,yes,31988.0,43.0,2015.0,4.0,7.200067,10.437164
2,0.0,F,1.0,76.0,41527.0,1.0,1.0,Own,50012.0,Thursday,...,24318.12282,23.0,multi_vehicle_unclear,yes,60876.0,NaN,2015.0,6.0,8.172179,9.615872
3,1.0,F,1.0,54.0,42099.0,1.0,1.0,Rent,20138.0,Saturday,...,36958.26656,23.0,multi_vehicle_unclear,yes,152772.0,32.0,2015.0,3.0,7.319163,9.740113
4,0.0,F,1.0,54.0,47206.0,1.0,1.0,Own,50033.0,Sunday,...,11779.17422,17.0,multi_vehicle_clear,yes,41151.0,31.0,2016.0,1.0,8.533387,10.748212


In [11]:
# Binning zero-inflated counts
df["past_claims_bin"] = pd.cut(
    df["past_num_of_claims"],
    bins=[-1, 0, 2, np.inf],
    labels=["0", "1-2", "3+"]
)
df.head()

,subrogation,gender,email_or_tel_available,safety_rating,annual_income,high_education_ind,address_change_ind,living_status,zip_code,claim_day_of_week,...,age_of_DL,accident_type,in_network_bodyshop,vehicle_mileage,driver_age,claim_year,claim_month,log_claim_est_payout,log_vehicle_price,past_claims_bin
0,1.0,F,0.0,75.0,70966.0,1.0,1.0,Rent,80040.0,Saturday,...,25.0,multi_vehicle_clear,no,75421.0,26.0,2016.0,12.0,8.077087,9.697270,3+
1,0.0,F,1.0,94.0,79723.0,1.0,1.0,Rent,80030.0,Wednesday,...,23.0,multi_vehicle_clear,yes,31988.0,43.0,2015.0,4.0,7.200067,10.437164,0
2,0.0,F,1.0,76.0,41527.0,1.0,1.0,Own,50012.0,Thursday,...,23.0,multi_vehicle_unclear,yes,60876.0,NaN,2015.0,6.0,8.172179,9.615872,1-2
3,1.0,F,1.0,54.0,42099.0,1.0,1.0,Rent,20138.0,Saturday,...,23.0,multi_vehicle_unclear,yes,152772.0,32.0,2015.0,3.0,7.319163,9.740113,0
4,0.0,F,1.0,54.0,47206.0,1.0,1.0,Own,50033.0,Sunday,...,17.0,multi_vehicle_clear,yes,41151.0,31.0,2016.0,1.0,8.533387,10.748212,3+


In [12]:
df['witness_present_ind'].head()

0    Y
1    N
2    N
3    N
4    N
Name: witness_present_ind, dtype: object

In [13]:
binary_map = {"Y": 1, "N": 0}
df["witness_present_ind"] = df["witness_present_ind"].map(binary_map)
df['witness_present_ind'].head()

0    1.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: witness_present_ind, dtype: float64

In [15]:
df.to_csv("/Users/eugene/Desktop/Emory/Projects/Travelers/data/processed/train_features.csv", index=False)